<a href="https://colab.research.google.com/github/ramyajeldy/Capstone_V2/blob/feature%2Fdataset/BERT_phishing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

!pip install evaluate # Install the missing library
import evaluate

In [5]:
dataset = load_dataset("drorrabin/phishing_emails-data")

dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


data/train-00000-of-00001.parquet:   0%|          | 0.00/11.0M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.88M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26946 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3705 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'email_type'],
        num_rows: 26946
    })
    test: Dataset({
        features: ['text', 'email_type'],
        num_rows: 3705
    })
})

In [6]:
print(dataset)
print(dataset["train"].column_names)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'email_type'],
        num_rows: 26946
    })
    test: Dataset({
        features: ['text', 'email_type'],
        num_rows: 3705
    })
})
['text', 'email_type']
{'text': 'Is the following email safe or phishing??\n\nDate: Thu, 07 Aug 2008 10:59:35 +0200\n\nSender: Daily Top 10 <avadivap_1981@techsult.com>\n\nReceiver: user2.12@gvc.ceas-challenge.cc\n\nEmail Subject: CNN.com Daily Top 10\n\nEmail Body: THE DAILY TOP 10 from CNN.com Top videos and stories as of: Aug  1, 2008  3:58 PM EDT  TOP 10 VIDEOS 1. PARIS HILTON TAKES ON MCCAIN http://www.cnn.com/video/partners/email/index.html?url/video/politics/2008/08/06/wynter.paris.hilton.ad.cnn Paris Hilton swings back at Republican presidential candidate John McCain. Kareen Wynter reports. 2. BIKINI BARISTA STAND CLOSED http://www.cnn.com/video/partners/email/index.html?url/video/living/2008/08/06/pkg.bikini.baristas.barred.kiro 3. TOT GRANDMA REACTS TO CHARGES http://www.cnn.c

In [7]:
dataset["train"].to_pandas()["email_type"].value_counts()

,count
email_type,
phishing email,13473
safe email,13473


In [8]:
def encode_labels(example):
    if example["email_type"] == "phishing email":
        example["labels"] = 1
    else:
        example["labels"] = 0
    return example

dataset = dataset.map(encode_labels)

Map:   0%|          | 0/26946 [00:00<?, ? examples/s]

Map:   0%|          | 0/3705 [00:00<?, ? examples/s]

In [9]:
dataset = dataset.remove_columns(["email_type"])

In [10]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [11]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/26946 [00:00<?, ? examples/s]

Map:   0%|          | 0/3705 [00:00<?, ? examples/s]

In [12]:

tokenized_dataset.set_format("torch")

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
tokenized_dataset["train"][0]

{'text': 'Is the following email safe or phishing??\n\nDate: Thu, 07 Aug 2008 10:59:35 +0200\n\nSender: Daily Top 10 <avadivap_1981@techsult.com>\n\nReceiver: user2.12@gvc.ceas-challenge.cc\n\nEmail Subject: CNN.com Daily Top 10\n\nEmail Body: THE DAILY TOP 10 from CNN.com Top videos and stories as of: Aug  1, 2008  3:58 PM EDT  TOP 10 VIDEOS 1. PARIS HILTON TAKES ON MCCAIN http://www.cnn.com/video/partners/email/index.html?url/video/politics/2008/08/06/wynter.paris.hilton.ad.cnn Paris Hilton swings back at Republican presidential candidate John McCain. Kareen Wynter reports. 2. BIKINI BARISTA STAND CLOSED http://www.cnn.com/video/partners/email/index.html?url/video/living/2008/08/06/pkg.bikini.baristas.barred.kiro 3. TOT GRANDMA REACTS TO CHARGES http://www.cnn.com/video/partners/email/index.html?url/video/crime/2008/08\n\nEmail type is: phishing email',
 'labels': tensor(1),
 'input_ids': tensor([  101,  2003,  1996,  2206, 10373,  3647,  2030, 13569, 12227,  1029,
          1029,  3

In [15]:
set(dataset["train"]["labels"])

{0, 1}

In [16]:
tokenized_dataset.set_format("torch")

In [17]:
tokenized_dataset["train"][0]

{'text': 'Is the following email safe or phishing??\n\nDate: Thu, 07 Aug 2008 10:59:35 +0200\n\nSender: Daily Top 10 <avadivap_1981@techsult.com>\n\nReceiver: user2.12@gvc.ceas-challenge.cc\n\nEmail Subject: CNN.com Daily Top 10\n\nEmail Body: THE DAILY TOP 10 from CNN.com Top videos and stories as of: Aug  1, 2008  3:58 PM EDT  TOP 10 VIDEOS 1. PARIS HILTON TAKES ON MCCAIN http://www.cnn.com/video/partners/email/index.html?url/video/politics/2008/08/06/wynter.paris.hilton.ad.cnn Paris Hilton swings back at Republican presidential candidate John McCain. Kareen Wynter reports. 2. BIKINI BARISTA STAND CLOSED http://www.cnn.com/video/partners/email/index.html?url/video/living/2008/08/06/pkg.bikini.baristas.barred.kiro 3. TOT GRANDMA REACTS TO CHARGES http://www.cnn.com/video/partners/email/index.html?url/video/crime/2008/08\n\nEmail type is: phishing email',
 'labels': tensor(1),
 'input_ids': tensor([  101,  2003,  1996,  2206, 10373,  3647,  2030, 13569, 12227,  1029,
          1029,  3

In [18]:
metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")
metric_precision = evaluate.load("precision")
metric_recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": metric_accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": metric_f1.compute(predictions=predictions, references=labels)["f1"],
        "precision": metric_precision.compute(predictions=predictions, references=labels)["precision"],
        "recall": metric_recall.compute(predictions=predictions, references=labels)["recall"],
    }

In [19]:
training_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="tensorboard"   # ✅ new way
)

In [20]:
torch.cuda.is_available()

True

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

In [22]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.004059,0.054605,0.985425,0.914557,0.976351,0.860119
2,0.006105,0.100090,0.980027,0.878689,0.978102,0.797619
3,0.000818,0.107017,0.982456,0.895330,0.975439,0.827381


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=5055, training_loss=0.00897104916646183, metrics={'train_runtime': 4255.4202, 'train_samples_per_second': 18.996, 'train_steps_per_second': 1.188, 'total_flos': 1.063468574659584e+16, 'train_loss': 0.00897104916646183, 'epoch': 3.0})

In [23]:
trainer.evaluate()

{'eval_loss': 0.05483486130833626,
 'eval_accuracy': 0.9856950067476383,
 'eval_f1': 0.9157392686804452,
 'eval_precision': 0.9829351535836177,
 'eval_recall': 0.8571428571428571,
 'eval_runtime': 60.9245,
 'eval_samples_per_second': 60.813,
 'eval_steps_per_second': 3.808,
 'epoch': 3.0}

In [24]:
trainer.state.best_model_checkpoint

'./bert_results/checkpoint-1685'

In [25]:
trainer.evaluate()

{'eval_loss': 0.05483486130833626,
 'eval_accuracy': 0.9856950067476383,
 'eval_f1': 0.9157392686804452,
 'eval_precision': 0.9829351535836177,
 'eval_recall': 0.8571428571428571,
 'eval_runtime': 60.8024,
 'eval_samples_per_second': 60.935,
 'eval_steps_per_second': 3.816,
 'epoch': 3.0}

In [26]:
predictions = trainer.predict(tokenized_dataset["test"])


In [27]:
import torch
import numpy as np
from torch.nn.functional import softmax

logits = torch.tensor(predictions.predictions)
probs = softmax(logits, dim=1).numpy()

phishing_probs = probs[:, 1]
true_labels = predictions.label_ids

In [28]:
from sklearn.metrics import f1_score

best_f1 = 0
best_threshold = 0.5

for t in np.arange(0.1, 0.9, 0.01):
    preds = (phishing_probs >= t).astype(int)
    f1 = f1_score(true_labels, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Best Threshold:", best_threshold)
print("Best F1:", best_f1)

Best Threshold: 0.11
Best F1: 0.9249617151607963


In [29]:
from sklearn.metrics import precision_score, recall_score

optimal_preds = (phishing_probs >= 0.11).astype(int)

print("Precision:", precision_score(true_labels, optimal_preds))
print("Recall:", recall_score(true_labels, optimal_preds))
print("F1:", f1_score(true_labels, optimal_preds))

Precision: 0.9526813880126183
Recall: 0.8988095238095238
F1: 0.9249617151607963


In [30]:
THRESHOLD = 0.12

def predict_email(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=1)

    phishing_prob = probs[0][1].item()

    return {
        "risk_score": round(phishing_prob, 4),
        "confidence": round(max(probs[0]).item(), 4),
        "label": "high_risk" if phishing_prob >= THRESHOLD else "low_risk"
    }

In [31]:
trainer.save_model("/content/phishing_bert_final")
tokenizer.save_pretrained("/content/phishing_bert_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/phishing_bert_final/tokenizer_config.json',
 '/content/phishing_bert_final/tokenizer.json')

In [32]:
import shutil
from google.colab import files

shutil.make_archive("phishing_bert_final", 'zip', "/content/phishing_bert_final")
files.download("phishing_bert_final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>